In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier

# =========================
# 1. Chargement des données
# =========================
labels = pd.read_csv("export.csv", usecols=["ID_PARCEL", "CODE_CULTU"])
labels = labels.dropna(subset=["CODE_CULTU"]).copy()
labels["ID_PARCEL"] = labels["ID_PARCEL"].astype(str)
labels["CODE_CULTU"] = labels["CODE_CULTU"].astype(str)
labels = labels.drop_duplicates(subset=["ID_PARCEL"])

indices = pd.read_csv("indices_parcelles_2024-01-01_2024-12-31_win5d.csv")
indices["ID_PARCEL"] = indices["ID_PARCEL"].astype(str)

print("Labels:", labels.shape)
print("Indices:", indices.shape)

# =========================
# 2. Filtrage des données
# =========================
indices = indices[indices["ID_PARCEL"].isin(labels["ID_PARCEL"])]
indices = indices[indices["index"].isin(["NDVI", "NDWI", "NDMI", "EVI"])]
indices = indices[indices["cloud_scene"] <= 40]
indices = indices[indices["px_count"] >= 10]
indices = indices.dropna(subset=["date", "ID_PARCEL", "index", "value_mean", "px_count", "cloud_scene"]).copy()

print("Indices après filtrage:", indices.shape)

# =========================
# 3. Agrégation pondérée
# =========================
indices["weighted"] = indices["value_mean"] * indices["px_count"]

agg = (
    indices.groupby(["ID_PARCEL", "date", "index"], as_index=False)
    .agg(weighted_sum=("weighted", "sum"),
         px_sum=("px_count", "sum"))
)

agg["value"] = agg["weighted_sum"] / agg["px_sum"]

print("Agrégation terminée :", agg.shape)
agg.head()

# =========================
# 4. Passage en format large
# =========================
agg["feature"] = agg["index"].astype(str) + "__" + agg["date"].astype(str)

wide = agg.pivot_table(
    index="ID_PARCEL",
    columns="feature",
    values="value",
    aggfunc="mean"
)

wide.columns.name = None
wide = wide.reset_index()

print("Wide shape:", wide.shape)
wide.head()

# =========================
# 5. Fusion avec les labels
# =========================
data = labels.merge(wide, on="ID_PARCEL", how="inner")

print("Data fusionnée:", data.shape)
print("Nombre de classes avant filtrage:", data["CODE_CULTU"].nunique())

# =========================
# 6. Filtrage des classes rares
# =========================
class_counts = data["CODE_CULTU"].value_counts()
keep_classes = class_counts[class_counts >= 50].index
data = data[data["CODE_CULTU"].isin(keep_classes)].copy()

print("Data après filtrage classes rares:", data.shape)
print("Nombre de classes après filtrage:", data["CODE_CULTU"].nunique())

# =========================
# 7. Préparation X / y
# =========================
X = data.drop(columns=["ID_PARCEL", "CODE_CULTU"]).copy()
y_str = data["CODE_CULTU"].astype(str)

missing_before = X.isna().mean().mean()
X = X.fillna(X.median(numeric_only=True))
missing_after = X.isna().mean().mean()

print(f"Missing avant imputation : {missing_before:.4f}")
print(f"Missing après imputation : {missing_after:.4f}")

encoder = LabelEncoder()
y = encoder.fit_transform(y_str)

# =========================
# 8. Séparation train / test
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

# =========================
# 9. Entraînement Random Forest
# =========================
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

# =========================
# 10. Prédictions
# =========================
y_pred = rf.predict(X_test)

# =========================
# 11. Évaluation
# =========================
acc = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")
weighted_f1 = f1_score(y_test, y_pred, average="weighted")

print("\n=== Résultats Random Forest ===")
print(f"Accuracy     : {acc:.4f}")
print(f"Macro-F1     : {macro_f1:.4f}")
print(f"Weighted-F1  : {weighted_f1:.4f}")

print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred, target_names=encoder.classes_, zero_division=0))

# =========================
# 12. Matrice de confusion
# =========================
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=encoder.classes_, columns=encoder.classes_)

print("\nMatrice de confusion :")
print(cm_df.head())

# =========================
# 13. Importance des variables
# =========================
importances = pd.DataFrame({
    "feature": X.columns,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)

print("\nTop 10 features les plus importantes :")
print(importances.head(10))



Labels: (150981, 2)
Indices: (11321544, 7)
Indices après filtrage: (11321544, 7)
Agrégation terminée : (11242568, 6)
Wide shape: (115712, 237)
Data fusionnée: (115712, 238)
Nombre de classes avant filtrage: 110
Data après filtrage classes rares: (114814, 238)
Nombre de classes après filtrage: 47
Missing avant imputation : 0.5884
Missing après imputation : 0.0000
Train shape: (91851, 236)
Test shape : (22963, 236)

=== Résultats Random Forest ===
Accuracy     : 0.6512
Macro-F1     : 0.3677
Weighted-F1  : 0.6278

=== Classification Report ===
              precision    recall  f1-score   support

         ARP       0.00      0.00      0.00        24
         AVH       0.75      0.15      0.25        40
         BDH       0.52      0.84      0.64       367
         BDP       1.00      0.36      0.53        11
         BOR       0.53      0.39      0.45      1689
         BTA       0.14      0.02      0.04       134
         BTH       0.54      0.42      0.47       195
         BTP       0